### Install the libraries

In [1]:
!pip install langsmith openinference-instrumentation-crewai openinference-instrumentation-groq -q

### Run CrewAI Agent

In [3]:
from crewai import Agent, Task, Crew, LLM, Process
from langsmith.integrations.otel import OtelSpanProcessor
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from openinference.instrumentation.crewai import CrewAIInstrumentor
from openinference.instrumentation.groq import GroqInstrumentor
from dotenv import load_dotenv
import os

load_dotenv()

tracer_provider = trace.get_tracer_provider()
if not isinstance(tracer_provider, TracerProvider):
    tracer_provider = TracerProvider()
    trace.set_tracer_provider(tracer_provider)

tracer_provider.add_span_processor(OtelSpanProcessor())

CrewAIInstrumentor().instrument()
GroqInstrumentor().instrument()

llm = LLM(model = "groq/llama-3.1-8b-instant", 
        temperature = 0.2,
        max_completion_tokens = 256,
        top_p = 0.9
    )

llm = LLM(model = "groq/llama-3.1-8b-instant", 
        temperature = 0.2,
        max_completion_tokens = 256,
        top_p = 0.9
    )

planner = Agent(
    role = "Content Planner", 
    goal = "Plan engaging and factually accurate content on {topic}", 
    backstory = "You're working on planning a blog article about the: {topic}"
                "You collect information that helps the audience learn something"
                "and make informed decisions."
                "Your work is the basis for the content writer to write an article on this topic.", 
    llm = llm, 
    allow_delegation = False, 
    verbose = True
)

writer = Agent(
    role = "Content Writer",
    goal = "Write a factually accurate opinion piece about the topic: {topic}",
    backstory = "You are working on writing a new opinion piece about the topic: {topic}."
                "You base your writing on the work of the content planner, who provides an outline"
                "You also provide objective and insights and back them up with the information"
                "provided by the content planner",
    llm = llm, 
    allow_delegation = False, 
    verbose = True
)

editor = Agent(
    role = "Editor", 
    goal = "Edit the given blog post with a proper writing style.",
    backstory = "You an editor who receives a blog post from the content writer."
                "Your goal is to ensure that it follows journalistic best practices.",
    llm = llm, 
    allow_delegation = False, 
    verbose = True
)   

plan = Task(
    description = (
        "1. Prioritize the latest trends and news on the {topic}.\n"
        "2. Identify the target audience based on their interests.\n"
        "3. Develop a detailed content outline including an introduction, and key points.\n"
        "4. Include SEO keywords and relevant data sources.\n"
    ),
    expected_output = "A comprehensive content plan document with an outline, audience analysis, SEO keywords, and resources.\n",
    agent = planner
)

write = Task(
    description = (
        "1. Use the content plan to craft a blog post on the {topic}.\n"
        "2. Incoporate SEO keywords naturally.\n"
        "3. Ensure the post is properly structured with an introduction, insightful body, and a summarising conclusion.\n"
        "4. Proofread for grammatical errors.\n"
    ),
    expected_output = "A well-written blog post in markdown format, ready for publication.", 
    agent = writer
)

edit = Task(
    description = (
        "Proofread the given blog post for grammatical errors and alignment with the core-opinion."
    ),
    expected_output = "A well-written blog post in markdown format, ready for publication.",
    agent = editor
)

crew = Crew(
    agents = [planner, writer, editor],
    tasks = [plan, write, edit],
    verbose = True, 
    tracing = True
)   

try: 
    result = crew.kickoff(inputs = {"topic": "Autonomous Enterprises"})
    print("Result:", result)
except Exception as e: 
    print(f"An error occured: {e}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 423b405c-864c-4c55-b733-be998b9236fa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends and news on the Autonomous Enterprises.                                  │
│  2. Identify the target audience based on their interests.                                                      │
│  3. Develop a detailed content outline including an introduction, and key points.                               │
│  4. Include SEO keywords and relevant data sources.                                                             │
│                                                                                                                 │
│  ID: 3a6b350e-faf9-4163-8078-e600f152425b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends and news on the Autonomous Enterprises.                                  │
│  2. Identify the target audience based on their interests.                                                      │
│  3. Develop a detailed content outline including an introduction, and key points.                               │
│  4. Include SEO keywords and relevant data sources.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Content Plan Document: Autonomous Enterprises**                                                              │
│                                                                                                                 │
│  **Task 1: Prioritize the latest trends and news on Autonomous Enterprises**                                    │
│                                                                                                                 │
│  1. **Rise of Autonomous Enterprises**: The concept of autonomous enterprises has gained significant attention  │
│  in recent years, with many companies adopting autonomous systems to improve efficiency and decision-making.    │
│  2. **Advancements in AI and Machine Learning**: The integration of AI and machine learning technologies has    │
│  enabled autonomous enterprises to make data-driven decisions, automate processes, and enhance customer         │
│  experiences.                                                                                                   │
│  3. **Increased Adoption of Cloud Computing**: Cloud computing has become a crucial enabler of autonomous       │
│  enterprises, providing scalable and flexible infrastructure for data processing and analytics.                 │
│  4. **Growing Importance of Cybersecurity**: As autonomous enterprises rely on interconnected systems,          │
│  cybersecurity has become a top concern, with companies investing in robust security measures to protect        │
│  against threats.                                                                                               │
│  5. **Emergence of Edge Computing**: Edge computing has emerged as a key technology for autonomous              │
│  enterprises, enabling real-time processing and analysis of data at the edge of the network.                    │
│                                                                                                                 │
│  **Task 2: Identify the target audience based on their interests**                                              │
│                                                                                                                 │
│  * **Primary Target Audience**: Business leaders, entrepreneurs, and decision-makers interested in adopting     │
│  autonomous technologies to improve their organizations' efficiency and competitiveness.                        │
│  * **Secondary Target Audience**: IT professionals, developers, and data scientists                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Prioritize the latest trends and news on the Autonomous Enterprises.                                  │
│  2. Identify the target audience based on their interests.                                                      │
│  3. Develop a detailed content outline including an introduction, and key points.                               │
│  4. Include SEO keywords and relevant data sources.                                                             │
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **The Rise of Autonomous Enterprises: Unlocking Efficiency and Innovation**                                    │
│  ================================================================================                               │
│                                                                                                                 │
│  ### Introduction                                                                                               │
│                                                                                                                 │
│  In today's fast-paced business landscape, companies are constantly seeking ways to stay ahead of the           │
│  competition. One trend that has gained significant attention in recent years is the rise of autonomous         │
│  enterprises. By leveraging cutting-edge technologies such as artificial intelligence (AI), machine learning    │
│  (ML), and cloud computing, autonomous enterprises are revolutionizing the way businesses operate, making       │
│  data-driven decisions, and enhancing customer experiences. In this article, we will delve into the latest      │
│  trends and news on autonomous enterprises, exploring their benefits, challenges, and the technologies that     │
│  are driving their adoption.                                                                                    │
│                                                                                                                 │
│  ### The Rise of Autonomous Enterprises                                                                         │
│                                                                                                                 │
│  The concept of autonomous enterprises has been gaining momentum in recent years, with many companies adopting  │
│  autonomous systems to improve efficiency and decision-making. According to a recent report, the global         │
│  autonomous enterprise market is expected to reach $1.4 trillion by 2025, growing at a compound annual growth   │
│  rate (CAGR) of 20.5% from 2020 to 2025. This rapid growth is driven by the increasing demand for automation,   │
│  AI, and ML technologies that can help businesses make data-driven decisions and improve customer experiences.  │
│                                                                                                                 │
│  ### Advancements in AI and Machine Learning                                                                    │
│                                                                                                                 │
│  The integration of AI and                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the core-opinion.                │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Result: **The Rise of Autonomous Enterprises: Unlocking Efficiency and Innovation**

### Introduction

In today's fast-paced business landscape, companies are constantly seeking ways to stay ahead of the competition. One trend that has gained significant attention in recent years is the rise of autonomous enterprises. By leveraging cutting-edge technologies such as artificial intelligence (AI), machine learning (ML), and cloud computing, autonomous enterprises are revolutionizing the way businesses operate, making data-driven decisions, and enhancing customer experiences. In this article, we will delve into the latest trends and news on autonomous enterprises, exploring their benefits, challenges, and the technologies that are driving their adoption.

### The Rise of Autonomous Enterprises

The concept of autonomous enterprises has been gaining momentum in recent years, with many companies adopting autonomous systems to improve efficiency and decision-making. According to a recent repo

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ b1791def-f0a0-40ec-befc-9ebdc5d710b6                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/b1791def-f0a0-40e │
│ c-befc-9ebdc5d710b6?access_code=TRACE-ff3a7d589c                             │
│ 🔑 Access Code: TRACE-ff3a7d589c                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 423b405c-864c-4c55-b733-be998b9236fa                                                                       │
│  Final Output: **The Rise of Autonomous Enterprises: Unlocking Efficiency and Innovation**                      │
│  ================================================================================                               │
│                                                                                                                 │
│  ### Introduction                                                                                               │
│                                                                                                                 │
│  In today's fast-paced business landscape, companies are constantly seeking ways to stay ahead of the           │
│  competition. One trend that has gained significant attention in recent years is the rise of autonomous         │
│  enterprises. By leveraging cutting-edge technologies such as artificial intelligence (AI), machine learning    │
│  (ML), and cloud computing, autonomous enterprises are revolutionizing the way businesses operate, making       │
│  data-driven decisions, and enhancing customer experiences. In this article, we will delve into the latest      │
│  trends and news on autonomous enterprises, exploring their benefits, challenges, and the technologies that     │
│  are driving their adoption.                                                                                    │
│                                                                                                                 │
│  ### The Rise of Autonomous Enterprises                                                                         │
│                                                                                                                 │
│  The concept of autonomous enterprises has been gaining momentum in recent years, with many companies adopting  │
│  autonomous systems to improve efficiency and decision-making. According to a recent report, the global         │
│  autonomous enterprise market is expected to reach $1.4 trillion by 2025, growing at a compound annual growth   │
│  rate (CAGR) of 20.5% from 2020 to 2025. This rapid growth is driven by the increasing demand for automation,   │
│  AI, and ML technologies that can help businesses make data-driven decisions and improve customer experiences.  │
│                                                                                                                 │
│  ### Advancements in AI and Machine Learning                                                                    │
│                                                                                                                 │
│  The integration of AI and                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯